# Reference workflow: region → crossmatch → gather → homogenize → ML export

**Environment:** [uv](https://docs.astral.sh/uv/) project venv (`uv sync --extra desi --extra dev`). Kernel **Python (data-lake)**.

This notebook walks through the full discovery pipeline on a real lake:

1. **Inventory** — what surveys/modalities exist
2. **Area** — save a sky region + crossmatch/gather/homogenize plan
3. **Discover** — tile overlap estimates in the region
4. **Crossmatch** — base × partner positional matches (region-bounded)
5. **Gather** — wide native product catalog
6. **Homogenize** — `phot_ab_v1` AB magnitudes on the gathered product
7. **ML export** — Parquet extract + optional calibrated spectra subset

Companion docs: [`docs/discovery/workflow.md`](../docs/discovery/workflow.md), [`docs/discovery/areas-cli.md`](../docs/discovery/areas-cli.md).  
Example area template: [`examples/areas/multi_survey_cone.example.json`](../examples/areas/multi_survey_cone.example.json).

> Use **`dl-area`** to attach `crossmatch_plan`, `gather`, and `homogenize` after `dl-region --save-as`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

# ── Set your deployment ────────────────────────────────────────────────────────
# Set DATA_LAKE_CONFIG before running this notebook, e.g.:
#   export DATA_LAKE_CONFIG=/path/to/your/lake_config.toml
#
# If the env var is already set (e.g. in your shell or via a .env file),
# the line below is a no-op.  Replace the placeholder path only if you want
# to hard-code a fallback for local development.
os.environ.setdefault("DATA_LAKE_CONFIG", os.environ.get("DATA_LAKE_CONFIG", ""))

from data_lake.config import LakeConfig

cfg = LakeConfig.discover()
LAKE_ROOT = Path(cfg.lake.root)
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data_lake").is_dir() and (REPO_ROOT.parent / "data_lake").is_dir():
    REPO_ROOT = REPO_ROOT.parent

AREA_ID = "multi_survey_cone"  # edit if you rename the example area
EXAMPLE_AREA = REPO_ROOT / "examples/areas/multi_survey_cone.example.json"
SCRATCH = Path("/scratch/data_lake_workflow")  # change to a writable path
SCRATCH.mkdir(parents=True, exist_ok=True)

print("Lake root:", LAKE_ROOT)
print("Config:", os.environ.get("DATA_LAKE_CONFIG"))

## 1. Inventory

Before defining an area, confirm survey names and column manifests match your lake (`dl-describe-lake` / `dl-describe-survey`).

In [ ]:
def run_dl(*args: str, check: bool = True) -> subprocess.CompletedProcess:
    cmd = ["dl-" + args[0]] + list(args[1:])
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=True)

print(run_dl("describe-lake", "--modality", "catalog").stdout[:4000])

In [ ]:
# Adjust survey names to match your lake before running gather/homogenize.
for survey in ("EUCLID_DR1", "DESI_DR1", "ALLWISE"):
    p = LAKE_ROOT / "catalogs" / survey
    if not p.is_dir():
        print(f"SKIP {survey} (not ingested)")
        continue
    out = run_dl("describe-survey", survey, "--modality", "catalog")
    print(out.stdout.splitlines()[0:8])
    print()

## 2. Install the example area

Option A — import the bundled template with **`dl-area`**:

```bash
dl-area import multi_survey_cone \
  --from-file examples/areas/multi_survey_cone.example.json --create-region
```

Option B — save region only, then attach plans step by step:

```bash
dl-region $LAKE --cone 150.1 2.2 --radius-arcsec 600 --save-as multi_survey_cone
dl-area set-crossmatch multi_survey_cone --base EUCLID_DR1 --partner DESI_DR1:1.0 ...
```

In [ ]:
from data_lake.discovery.areas import load_area, validate_area

areas_dir = LAKE_ROOT / "areas"
areas_dir.mkdir(parents=True, exist_ok=True)
dest = areas_dir / f"{AREA_ID}.json"

if not dest.exists():
    # Prefer dl-area import; fall back to file copy for notebook-only runs
    try:
        run_dl("area", str(LAKE_ROOT), "import", AREA_ID,
               "--from-file", str(EXAMPLE_AREA), "--create-region")
    except Exception:
        shutil.copy(EXAMPLE_AREA, dest)
        print("Copied example area →", dest)
else:
    print("Area already exists:", dest)

area = load_area(LAKE_ROOT, AREA_ID)
msgs = validate_area(area.data)
for m in msgs:
    print(m)
print(json.dumps(area.data, indent=2)[:2000], "…")

## 3. Discover tile overlap

Index-driven estimates — no tile reads unless you pass `count=True` for exact catalog row sums.

In [ ]:
from data_lake.discovery.engine import resolve_region, round_count
from data_lake.schema_registry import MODALITY_CATALOG, MODALITY_CUTOUT, MODALITY_SPECTRA

rows = resolve_region(
    LAKE_ROOT, area.region, surveys="all",
    modalities=[MODALITY_CATALOG, MODALITY_SPECTRA, MODALITY_CUTOUT],
)
for r in rows[:12]:
    est = round_count(r.est_rows)
    print(f"{r.survey:20s} {r.modality:10s}  tiles={r.n_tiles_overlap:4d}  est_rows={est}")
if len(rows) > 12:
    print(f"… and {len(rows) - 12} more rows")

## 4. Crossmatch (base × partners)

Requires `crossmatch_plan` in the area file. Re-runs are resumable (`reuse_existing: true`).

In [ ]:
# Uncomment to run on your lake (can take minutes–hours depending on region size).
# result = run_dl("crossmatch", str(LAKE_ROOT), "--from-area", AREA_ID, "--n-workers", "4", "--progress")
# print(result.stdout)

from data_lake.io.crossmatch import load_crossmatch_plan
plan, region = load_crossmatch_plan(LAKE_ROOT, from_area=AREA_ID)
print("base:", plan["base_catalog"])
print("partners:", [p["survey"] for p in plan["partners"]])

In [ ]:
# Python API equivalent (same as dl-crossmatch --from-area)
from data_lake.io.crossmatch import execute_crossmatch_plan

# results = execute_crossmatch_plan(
#     LAKE_ROOT, plan, region=region, n_workers=4, show_progress=True,
# )
# for r in results:
#     print(r.crossmatch_name, r.n_match_rows, "rows")

## 5. Gather — wide native product

Joins partner columns via existing crossmatch trees. Output lives under `catalogs/<materialize_as>/` with `kind: product`.

In [ ]:
gather_block = area.gather or {}
PRODUCT_NATIVE = gather_block.get("materialize_as", "euclid_north_native_v1")

# result = run_dl("gather", str(LAKE_ROOT), "--from-area", AREA_ID, "--progress")
# print(result.stdout)

print("Would materialise:", PRODUCT_NATIVE)
print("columns:", json.dumps(gather_block.get("columns", {}), indent=2))

## 6. Homogenize — `phot_ab_v1` on the gathered product

Homogenization maps native Vega/mm/Jy columns to comparable AB magnitudes. Recipes live in `data_lake/homogenize/surveys/<SURVEY>.json` (or lake overrides).

Run `dl-validate-homogenization --ab-coverage` to find surveys still missing recipes.

In [ ]:
hom_block = area.homogenize or {}
PRODUCT_AB = hom_block.get("materialize_as", "euclid_north_ab_v1")
FROM_PRODUCT = hom_block.get("from_product", PRODUCT_NATIVE)

# Dry run — rule resolution only
# run_dl("homogenize", "--from-product", FROM_PRODUCT, "--transform", "phot_ab_v1",
#        "--materialize-as", PRODUCT_AB, "--from-area", AREA_ID, "--check-only")

from data_lake.discovery.selection import selection_from_region
from data_lake.homogenize.engine import homogenize_product

selection = selection_from_region(LAKE_ROOT, FROM_PRODUCT, area.region)

# check = homogenize_product(
#     LAKE_ROOT, FROM_PRODUCT, "phot_ab_v1", selection,
#     materialize_as=PRODUCT_AB, check_only=True,
# )
# print("Rules applied:", len(check.resolution.get("applied", [])))

In [ ]:
# Materialise homogenized product (uncomment when crossmatch + gather exist)
# result = homogenize_product(
#     LAKE_ROOT, FROM_PRODUCT, "phot_ab_v1", selection,
#     materialize_as=PRODUCT_AB, overwrite=False, show_progress=True,
# )
# print(result.n_rows, "rows →", result.output_root)

# Or use the area homogenize block:
# run_dl("homogenize", "--from-area", AREA_ID)

## 7. ML export — homogenized Parquet + provenance

`dl-extract-catalog --from-product` writes `extract_provenance.json` (transform id + column lineage) beside the export.

In [ ]:
from data_lake.export.catalog_extract import extract_catalog
from data_lake.io.catalog import CatalogAccessor

out_parquet = SCRATCH / f"{PRODUCT_AB}.parquet"
product_root = LAKE_ROOT / "catalogs" / PRODUCT_AB

if product_root.is_dir():
    acc = CatalogAccessor(LAKE_ROOT, PRODUCT_AB)
    # Pick columns that exist on your product (prefixed partner names)
    cols = [c for c in acc.columns if c in {
        "_source_id", "ra", "dec", "ALLWISE_phot_ab_w1", "ALLWISE_phot_ab_w2",
        "DESI_DR1_z", "DESI_DR1_sep_arcsec",
    }]
    if not cols:
        cols = ["_source_id", "ra", "dec"]
    result = extract_catalog(
        lake_root=LAKE_ROOT,
        survey=PRODUCT_AB,
        specs=cols,
        output=out_parquet,
        output_format="parquet",
        require_homogenized=True,
    )
    print(result)
    prov = out_parquet.with_suffix(".homogenize_provenance.json")
    if prov.is_file():
        print(prov.read_text()[:500])
else:
    print(f"Product not found yet: {product_root}")
    print("Run gather + homogenize first, or point PRODUCT_AB at an existing product.")

## 8. Optional — spectra for the same selection

Two paths:

1. **`dl-gather --extract-modalities spectra`** — bundle driven from the gathered product's `_source_id` list.
2. **`dl-extract-spectra-subset`** — direct extract from a target list; use `--apply-survey-calibration` for SDSS/DESI flux units (see [`export-and-sharing.md`](../docs/export-and-sharing.md)).

In [ ]:
spectra_dir = SCRATCH / "spectra_subset"
spectra_dir.mkdir(parents=True, exist_ok=True)

# Bundle via gather (after native product exists):
# run_dl("gather", str(LAKE_ROOT), "--from-area", AREA_ID,
#        "--extract-modalities", "spectra", "--output-dir", str(spectra_dir),
#        "--extract-survey", "DESI_DR1")

# Or extract a small ID list with flux calibration:
# from data_lake.io.catalog import CatalogAccessor
# from data_lake.io.spectra import SpectrumAccessor
# cat = CatalogAccessor(LAKE_ROOT, "DESI_DR1")
# ids = cat.query("SELECT _source_id FROM catalog WHERE _spectrum_index >= 0 LIMIT 100")["_source_id"].to_numpy()
# ids_file = SCRATCH / "desi_ids.txt"
# ids_file.write_text("\n".join(map(str, ids)))
# run_dl("extract-spectra-subset", "--survey", "DESI_DR1", "--target-list", str(ids_file),
#        "--lake-root", str(LAKE_ROOT), "--apply-survey-calibration",
#        "--output", str(spectra_dir / "desi_cal.zarr"))

## Checklist

| Step | Verify |
|------|--------|
| Area valid | `validate_area` → no `ERROR` lines |
| Crossmatch trees | `dl-describe-lake` or `crossmatch/` dirs exist for each partner |
| Gather product | `dl-describe-lake --kind product` lists `<materialize_as>` |
| Homogenize | `dl-validate-homogenization --product <ab_product>` |
| ML export | Parquet row count + `*.homogenize_provenance.json` |

**Next:** extend `dl-area` (e.g. `set-discover`, partner modality flags) as needed.